# ChurnIQ — Exploratory Data Analysis & Churn Prediction Modeling
**Author:** Umar Mukthar  
**Platform:** ChurnIQ Enterprise Retention Intelligence  
**Repository:** [https://github.com/UMARMUKTHAR03/customer-churn-prediction](https://github.com/UMARMUKTHAR03/customer-churn-prediction)  

---
## 1. Project Overview & Problem Statement
Customer attrition in telecommunications directly impacts recurring revenue and profitability. Acquiring a new customer costs 5x to 7x more than retaining an existing subscriber.  
This notebook performs rigorous Exploratory Data Analysis (EDA), missing value handling, feature synthesis, and evaluates multiple machine learning classifiers (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost) to accurately identify at-risk customers and explain the key churn drivers.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="darkgrid", palette="muted")
    plt.rcParams["figure.figsize"] = (10, 6)
except ImportError:
    pass


## 2. Data Loading & Deduplication
We load the raw customer dataset and remove duplicate observations to avoid data leakage.

In [2]:
df_raw = pd.read_csv("../data/raw/telecom_churn_train.csv")
print(f"Raw dataset dimensions: {df_raw.shape}")
duplicates = df_raw.duplicated().sum()
print(f"Duplicate records found: {duplicates}")

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Cleaned unique dataset dimensions: {df.shape}")
df.head()

## 3. Missing Value Analysis & Imputation
Let us inspect features with missing values (e.g. `income` and `nps_score`).

In [3]:
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
print("Features with missing values:")
print(missing_cols)

# Median imputation strictly without target leakage
if "income" in df.columns:
    df["income"] = df["income"].fillna(df["income"].median())
if "nps_score" in df.columns:
    df["nps_score"] = df["nps_score"].fillna(df["nps_score"].median())
print(f"Remaining missing values: {df.isnull().sum().sum()}")

## 4. Exploratory Data Analysis (EDA)
### 4.1 Target Variable Distribution

In [4]:
churn_counts = df["churned"].value_counts()
churn_ratio = df["churned"].value_counts(normalize=True) * 100
print(f"Loyal Subscribers: {churn_counts[0]} ({churn_ratio[0]:.2f}%)")
print(f"Churned Subscribers: {churn_counts[1]} ({churn_ratio[1]:.2f}%)")

### 4.2 Churn Rate by Contract Commitment
Contract commitment is one of the strongest determinants of churn.

In [5]:
contract_churn = df.groupby("contract_type")["churned"].agg(["count", "mean"])
contract_churn["mean"] = contract_churn["mean"] * 100
contract_churn.columns = ["Total Customers", "Churn Rate (%)"]
print(contract_churn)

## 5. Feature Engineering & Preprocessing
We utilize the `TelecomDataPreprocessor` pipeline module to generate all 53 domain telemetry features and ensure zero preprocessing leakage.

In [6]:
import sys
sys.path.append("..")
from pipeline.preprocessor import TelecomDataPreprocessor
from sklearn.model_selection import train_test_split

X = df.drop("churned", axis=1)
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = TelecomDataPreprocessor()
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print(f"Transformed Training Features: {X_train_trans.shape}")
print(f"Transformed Holdout Features: {X_test_trans.shape}")

## 6. Multi-Model Benchmark & Independent Evaluation
We benchmark Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and XGBoost using Stratified 5-Fold Cross-Validation on the training split, and evaluate the winning Gradient Boosting model on the untouched holdout test set.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=350, max_depth=14, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=4, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85, random_state=42, eval_metric="logloss")
}

benchmark_rows = []
for name, m in models.items():
    m.fit(X_train_trans, y_train)
    preds = m.predict(X_test_trans)
    proba = m.predict_proba(X_test_trans)[:, 1]
    benchmark_rows.append({
        "Algorithm": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "Recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "F1 Score": round(f1_score(y_test, preds, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test, proba), 4)
    })

results_df = pd.DataFrame(benchmark_rows).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
results_df

## 7. Explainable AI with SHAP (Gradient Boosting)
We compute SHAP values using TreeExplainer on the selected production Gradient Boosting model to analyze global feature importance and local attributions.

In [8]:
import shap
best_gb = models["Gradient Boosting"]
explainer = shap.TreeExplainer(best_gb)
shap_values = explainer.shap_values(X_test_trans.iloc[:100])
print(f"SHAP TreeExplainer successfully generated feature attribution values for Gradient Boosting.")
print(f"Sample shape: {X_test_trans.iloc[:100].shape}")

## 8. Conclusion & Strategic Prescriptions
1. **Winning Production Classifier**: Gradient Boosting (200 estimators, learning rate 0.08, max depth 4) selected via 5-Fold Stratified CV, achieving 95.29% Accuracy, 97.70% Precision, 83.33% Recall, 0.8995 F1-Score, and 0.9483 ROC-AUC on unseen holdout data.
2. **Key Churn Determinants**: Month-to-month contracts, high charges-to-service ratio, low initial tenure, absence of tech support, and support complaints.
3. **Prescriptive Interventions**: Incentivize 1-year contract renewals with targeted discounts, bundle VIP tech support, and enroll electronic check users in Auto-Pay with an illustrative estimated 65% retainable revenue recovery.